# Mini-GPT End-to-End (Single Colab Notebook)

This notebook is designed for a full presentation flow:

1. Use Google Colab GPU.
2. Choose and inspect candidate datasets.
3. Train a Hugging Face BPE tokenizer.
4. Build Mini-GPT transformer architecture from scratch.
5. Train, validate, test, and compute perplexity.
6. Save and reload model/tokenizer for live demo inference.

The notebook is intentionally readable and presentation-friendly.

## 1. Colab GPU Setup and Dependency Installation

**Before running this notebook on Google Colab:**

1. Go to `Runtime → Change runtime type → T4 GPU` (free tier) or `A100` (Colab Pro).
2. Then click `Runtime → Run all`, or run cells one by one from top to bottom.

> **Important:** This notebook uses **streaming mode** to load OpenWebText — it will NOT download the full 40GB dataset. Only the tokens you actually need are fetched.

This cell installs missing packages automatically and checks GPU availability.


In [ ]:
import importlib.util
import os
import platform
import subprocess
import sys

required_modules = {
    "datasets": "datasets",
    "tokenizers": "tokenizers",
    "transformers": "transformers",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
}

missing_packages = [pkg for mod, pkg in required_modules.items() if importlib.util.find_spec(mod) is None]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_packages)

import contextlib
import json
import math
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, processors, trainers
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm.auto import tqdm
from transformers import PreTrainedTokenizerFast

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (torch): {torch.version.cuda}")
    _ = os.system("nvidia-smi")

## 2. Global Config, Paths, and Reproducibility

All important settings are collected in one config dictionary so you can explain and tune experiments quickly during presentation.

In [ ]:
CONFIG = {
    "seed": 42,
    # OpenWebText mirrors GPT-2 training data: diverse news, tech, science, blogs.
    # It produces contextually grounded outputs instead of TinyStories "Once upon a time" mode.
    "selected_dataset_id": "openwebtext",
    "max_train_samples": 100000,
    "max_valid_samples": 3000,
    "max_test_samples": 3000,
    # Larger vocab handles web-domain subword diversity better.
    "vocab_size": 16000,
    "context_length": 256,
    # Stride = context_length // 2 balances training coverage and runtime.
    "sequence_stride": 128,
    "batch_size": 16,
    "grad_accum_steps": 2,
    "num_layers": 6,
    "num_heads": 6,
    "embedding_dim": 384,
    "dropout": 0.1,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "epochs": 3,
    # Keep this None to use epoch-based training, or set an int to force-stop early.
    "max_steps": None,
    "eval_interval": 100,
    "eval_batches": 30,
    "warmup_steps": 200,
    "grad_clip": 1.0,
    # ~8M tokens on web text trains a coherent 20M model in 1–1.5 hours on T4.
    "max_train_tokens": 8_000_000,
    "max_val_tokens": 400_000,
    "max_test_tokens": 400_000,
    "num_workers": 2,
}

DATASET_CANDIDATES = [
    {
        "id": "openwebtext",
        "path": "Skylion007/openwebtext",
        "name": None,
        "split": "train",
        "text_key": "text",
    },
    {
        "id": "tinystories",
        "path": "roneneldan/TinyStories",
        "name": None,
        "split": "train",
        "text_key": "text",
    },
    {
        "id": "wikitext103",
        "path": "wikitext",
        "name": "wikitext-103-raw-v1",
        "split": "train",
        "text_key": "text",
    },
]

ROOT = Path("/kaggle/working/mini_gpt_demo") if Path("/kaggle/working").exists() else Path("/content/mini_gpt_colab_demo")
TOKENIZER_DIR = ROOT / "tokenizer"
CHECKPOINT_DIR = ROOT / "checkpoints"
ARTIFACT_DIR = ROOT / "artifacts"

for path in [ROOT, TOKENIZER_DIR, CHECKPOINT_DIR, ARTIFACT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print("Working directory:", ROOT)
print("Seed:", CONFIG["seed"])
print("Selected dataset ID:", CONFIG["selected_dataset_id"])
print(
    "Model size: layers={num_layers}, heads={num_heads}, emb={embedding_dim}, ctx={context_length}".format(**CONFIG)
)
print(
    "Training plan: epochs={epochs}, batch_size={batch_size}, grad_accum_steps={grad_accum_steps}, stride={sequence_stride}".format(**CONFIG)
)
print(
    "Token budget: train={max_train_tokens:,}, val={max_val_tokens:,}, test={max_test_tokens:,}".format(**CONFIG)
)


## 3. Dataset Candidate Evaluation and Final Selection

We compare practical datasets for a small GPT demo:

| Dataset | Domain | Best for |
|---|---|---|
| **OpenWebText** | News, tech, science, blogs (mirrors GPT-2 training) | Contextually accurate, topic-aware generation |
| TinyStories | Simple children's stories | Short creative text, but always outputs "Once upon a time" |
| wikitext-103 | Wikipedia | Factual but dry; poor creative generation quality |

This notebook defaults to **OpenWebText** (`Skylion007/openwebtext`) because:
- Your prompts involve AI, smart cities, and astronauts — web text domain produces relevant continuations.
- TinyStories pulls every prompt into "Once upon a time" mode regardless of the topic.
- OpenWebText is the same corpus used to train GPT-2, so it is well-suited for this architecture.


In [ ]:
def load_candidate_preview(candidate: dict, n: int = 100) -> dict:
    # streaming=True avoids downloading the full dataset just for a preview.
    ds = load_dataset(candidate["path"], candidate["name"], split=candidate["split"], streaming=True)
    texts = []
    for sample in ds:
        text = str(sample.get(candidate["text_key"], "")).strip()
        if text:
            texts.append(text)
        if len(texts) >= n:
            break

    if not texts:
        return {
            "dataset_id": candidate["id"],
            "samples_used": 0,
            "avg_chars": 0,
            "avg_words": 0,
            "sample_preview": "No text found",
        }

    avg_chars = sum(len(t) for t in texts) / len(texts)
    avg_words = sum(len(t.split()) for t in texts) / len(texts)

    return {
        "dataset_id": candidate["id"],
        "samples_used": len(texts),
        "avg_chars": round(avg_chars, 2),
        "avg_words": round(avg_words, 2),
        "sample_preview": texts[0][:200].replace("\n", " "),
    }

comparison_rows = []
for candidate in DATASET_CANDIDATES:
    print(f"Evaluating {candidate['id']}...")
    row = load_candidate_preview(candidate, n=100)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

selected_meta = next(c for c in DATASET_CANDIDATES if c["id"] == CONFIG["selected_dataset_id"])
print("Final selected dataset:", selected_meta["id"], "->", selected_meta["path"])


## 4. Dataset Download and Loading Pipeline

We load the selected dataset using **streaming mode** — this is essential for Colab because OpenWebText is ~40GB and would exceed Colab's disk space if downloaded in full. Streaming fetches only the samples we actually need.


In [ ]:
def load_texts_streaming(path, name, split, text_key, max_samples, desc) -> list[str]:
    """Load text samples via HuggingFace streaming — no full dataset download required.

    This is the correct approach for Colab because large datasets like OpenWebText
    (40GB+ raw) would exhaust Colab's 15GB disk if loaded normally.
    Instead, we iterate the stream and stop as soon as we have enough samples.
    """
    ds = load_dataset(path, name, split=split, streaming=True)
    texts = []
    for sample in tqdm(ds, desc=desc, total=max_samples):
        text = str(sample.get(text_key, "")).strip()
        if text:
            texts.append(text)
        if len(texts) >= max_samples:
            break
    return texts


# Peek at available splits (streaming is safe here — no data is downloaded yet).
ds_info = load_dataset(selected_meta["path"], selected_meta["name"], streaming=True)
available_splits = list(ds_info.keys())
print("Available splits:", available_splits)

train_texts_raw = load_texts_streaming(
    selected_meta["path"],
    selected_meta["name"],
    "train",
    selected_meta["text_key"],
    CONFIG["max_train_samples"],
    desc=f"Streaming {selected_meta['id']} train",
)

if "validation" in available_splits:
    val_plus_test_raw = load_texts_streaming(
        selected_meta["path"],
        selected_meta["name"],
        "validation",
        selected_meta["text_key"],
        CONFIG["max_valid_samples"] + CONFIG["max_test_samples"],
        desc=f"Streaming {selected_meta['id']} validation",
    )
else:
    # OpenWebText only has a train split — carve validation/test from the end of train.
    fallback_needed = CONFIG["max_valid_samples"] + CONFIG["max_test_samples"]
    val_plus_test_raw = train_texts_raw[-fallback_needed:]
    train_texts_raw = train_texts_raw[:-fallback_needed]
    print("No validation split found — carved last", fallback_needed, "samples from train.")

print(f"Raw train texts: {len(train_texts_raw)}")
print(f"Raw val+test texts: {len(val_plus_test_raw)}")


## 5. Text Cleaning and Train/Validation/Test Split

We apply minimal cleaning and create stable train/validation/test text lists for language modeling.

In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_corpus(texts: list[str]) -> list[str]:
    cleaned = []
    for text in texts:
        c = clean_text(text)
        if c:
            cleaned.append(c)
    return cleaned

train_texts = clean_corpus(train_texts_raw)
val_plus_test = clean_corpus(val_plus_test_raw)

val_size = min(CONFIG["max_valid_samples"], len(val_plus_test))
val_texts = val_plus_test[:val_size]
remaining = val_plus_test[val_size:]

test_size = min(CONFIG["max_test_samples"], len(remaining))
test_texts = remaining[:test_size]

if len(test_texts) == 0:
    test_texts = val_texts[: min(1000, len(val_texts))]

print(f"Clean train: {len(train_texts)}")
print(f"Clean val: {len(val_texts)}")
print(f"Clean test: {len(test_texts)}")
print("Sample cleaned text:")
print(train_texts[0][:300])

## 6. Hugging Face Tokenizer API: Train BPE Tokenizer

We train a BPE tokenizer using Hugging Face Tokenizers directly on our training text.

Special tokens are configured for causal language modeling: `[PAD]`, `[UNK]`, `[BOS]`, `[EOS]`.

In [ ]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

base_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
base_tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC()])
base_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)

bpe_trainer = trainers.BpeTrainer(
    vocab_size=CONFIG["vocab_size"],
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
)


def bpe_training_iterator(texts: list[str], batch_size: int = 1000):
    for i in range(0, len(texts), batch_size):
        yield texts[i : i + batch_size]


base_tokenizer.train_from_iterator(
    bpe_training_iterator(train_texts),
    trainer=bpe_trainer,
    length=len(train_texts),
)

base_tokenizer.decoder = decoders.ByteLevel()

bos_id = base_tokenizer.token_to_id("[BOS]")
eos_id = base_tokenizer.token_to_id("[EOS]")

base_tokenizer.post_processor = processors.TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[("[BOS]", bos_id), ("[EOS]", eos_id)],
)

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=base_tokenizer,
    bos_token="[BOS]",
    eos_token="[EOS]",
    unk_token="[UNK]",
    pad_token="[PAD]",
)

hf_tokenizer.save_pretrained(TOKENIZER_DIR.as_posix())

print("Tokenizer saved to:", TOKENIZER_DIR)
print("Tokenizer vocab size:", hf_tokenizer.vocab_size)
print("Special token ids:")
print("  PAD:", hf_tokenizer.pad_token_id)
print("  UNK:", hf_tokenizer.unk_token_id)
print("  BOS:", hf_tokenizer.bos_token_id)
print("  EOS:", hf_tokenizer.eos_token_id)

## 7. Tokenizer Validation and Encoding Diagnostics

We perform sanity checks to verify encoding/decoding behavior and inspect sequence-length statistics.

In [ ]:
sample_text = train_texts[0][:240]
encoded = hf_tokenizer(sample_text, add_special_tokens=True)
decoded = hf_tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)

print("Original sample:")
print(sample_text)
print("\nEncoded token ids (first 40):")
print(encoded["input_ids"][:40])
print("\nDecoded sample:")
print(decoded[:240])


def average_token_length(texts: list[str], max_samples: int = 1000) -> float:
    lengths = []
    for text in texts[: max_samples]:
        lengths.append(len(hf_tokenizer.encode(text, add_special_tokens=True)))
    return float(sum(lengths) / max(1, len(lengths)))

print("\nAverage token lengths:")
print("Train:", round(average_token_length(train_texts), 2))
print("Val  :", round(average_token_length(val_texts), 2))
print("Test :", round(average_token_length(test_texts), 2))

unknown_example = "ThisContainsSomeRare$$$Pattern"
unknown_ids = hf_tokenizer.encode(unknown_example, add_special_tokens=True)
print("\nUnknown token behavior check IDs (first 20):", unknown_ids[:20])

## 8. Causal LM Sequence Packing and DataLoaders

We concatenate tokenized text and create fixed-length `(input, target)` pairs with a one-token shift.

Important runtime note:
Using a sliding window stride of `1` creates an extremely large number of samples and can make Kaggle training take tens of hours. This notebook uses a configurable `sequence_stride` so training remains practical.

In [ ]:
def tokenize_and_flatten(
    texts: list[str],
    tokenizer: PreTrainedTokenizerFast,
    max_total_tokens: int,
) -> list[int]:
    token_stream = []
    for text in tqdm(texts, desc="Tokenizing"):
        ids = tokenizer.encode(text, add_special_tokens=True)
        token_stream.extend(ids)
        if len(token_stream) >= max_total_tokens:
            token_stream = token_stream[:max_total_tokens]
            break
    return token_stream


train_tokens = tokenize_and_flatten(train_texts, hf_tokenizer, CONFIG["max_train_tokens"])
val_tokens = tokenize_and_flatten(val_texts, hf_tokenizer, CONFIG["max_val_tokens"])
test_tokens = tokenize_and_flatten(test_texts, hf_tokenizer, CONFIG["max_test_tokens"])

print(f"Train tokens: {len(train_tokens):,}")
print(f"Val tokens:   {len(val_tokens):,}")
print(f"Test tokens:  {len(test_tokens):,}")


class PackedCausalDataset(TorchDataset):
    def __init__(self, token_ids: list[int], context_length: int, stride: int) -> None:
        if len(token_ids) <= context_length:
            raise ValueError("Not enough tokens for the given context_length")
        if stride <= 0:
            raise ValueError("stride must be a positive integer")

        self.token_ids = token_ids
        self.context_length = context_length
        self.stride = stride
        self.start_positions = list(range(0, len(token_ids) - context_length, stride))

    def __len__(self) -> int:
        return len(self.start_positions)

    def __getitem__(self, idx: int):
        start = self.start_positions[idx]
        x = self.token_ids[start : start + self.context_length]
        y = self.token_ids[start + 1 : start + self.context_length + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


train_dataset = PackedCausalDataset(train_tokens, CONFIG["context_length"], CONFIG["sequence_stride"])
val_dataset = PackedCausalDataset(val_tokens, CONFIG["context_length"], CONFIG["sequence_stride"])
test_dataset = PackedCausalDataset(test_tokens, CONFIG["context_length"], CONFIG["sequence_stride"])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)

print("Train sequences:", len(train_dataset))
print("Val sequences:", len(val_dataset))
print("Test sequences:", len(test_dataset))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 9. Transformer Building Blocks from Scratch

We implement decoder-only transformer components manually:

- Token + positional embeddings
- Causal masked multi-head self-attention
- Feed-forward network
- Residual + layer normalization

In [ ]:
@dataclass
class MiniGPTConfig:
    vocab_size: int
    context_length: int
    embedding_dim: int
    num_heads: int
    num_layers: int
    dropout: float


class TokenPositionalEmbedding(nn.Module):
    def __init__(self, cfg: MiniGPTConfig):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.embedding_dim)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.embedding_dim)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        bsz, seq_len = input_ids.shape
        del bsz
        positions = torch.arange(seq_len, device=input_ids.device)
        return self.dropout(self.token_emb(input_ids) + self.pos_emb(positions))


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: MiniGPTConfig):
        super().__init__()
        if cfg.embedding_dim % cfg.num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads")

        self.num_heads = cfg.num_heads
        self.head_dim = cfg.embedding_dim // cfg.num_heads
        self.embedding_dim = cfg.embedding_dim

        self.qkv_proj = nn.Linear(cfg.embedding_dim, 3 * cfg.embedding_dim)
        self.out_proj = nn.Linear(cfg.embedding_dim, cfg.embedding_dim)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

        mask = torch.tril(torch.ones(cfg.context_length, cfg.context_length))
        self.register_buffer("causal_mask", mask.view(1, 1, cfg.context_length, cfg.context_length))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        mask = self.causal_mask[:, :, :seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, self.embedding_dim)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, cfg: MiniGPTConfig):
        super().__init__()
        hidden_dim = 4 * cfg.embedding_dim
        self.net = nn.Sequential(
            nn.Linear(cfg.embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, cfg.embedding_dim),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: MiniGPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.embedding_dim)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.embedding_dim)
        self.ff = FeedForward(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

## 10. Mini-GPT Assembly and Parameter Inspection

We assemble the decoder-only model and verify output tensor shapes and parameter count.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, cfg: MiniGPTConfig):
        super().__init__()
        self.cfg = cfg
        self.embedding = TokenPositionalEmbedding(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.num_layers)])
        self.final_ln = nn.LayerNorm(cfg.embedding_dim)
        self.lm_head = nn.Linear(cfg.embedding_dim, cfg.vocab_size)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor | None = None):
        if input_ids.size(1) > self.cfg.context_length:
            raise ValueError("Input sequence length exceeds context length")

        x = self.embedding(input_ids)
        for block in self.blocks:
            x = block(x)
        x = self.final_ln(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @staticmethod
    def _get_banned_tokens_from_ngrams(tokens: list[int], ngram_size: int) -> set[int]:
        if ngram_size <= 1 or len(tokens) < ngram_size - 1:
            return set()

        prefix = tuple(tokens[-(ngram_size - 1):])
        banned = set()
        for idx in range(len(tokens) - ngram_size + 1):
            ngram = tokens[idx : idx + ngram_size]
            if tuple(ngram[:-1]) == prefix:
                banned.add(ngram[-1])
        return banned

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
        repetition_penalty: float = 1.15,
        no_repeat_ngram_size: int = 3,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            context = input_ids[:, -self.cfg.context_length :]
            logits, _ = self(context)
            next_logits = logits[:, -1, :] / max(temperature, 1e-8)

            if repetition_penalty is not None and repetition_penalty > 1.0:
                for b in range(input_ids.size(0)):
                    seen_tokens = torch.unique(input_ids[b])
                    next_logits[b, seen_tokens] = next_logits[b, seen_tokens] / repetition_penalty

            if no_repeat_ngram_size is not None and no_repeat_ngram_size > 1:
                for b in range(input_ids.size(0)):
                    banned = self._get_banned_tokens_from_ngrams(
                        input_ids[b].tolist(),
                        no_repeat_ngram_size,
                    )
                    if banned:
                        next_logits[b, list(banned)] = float("-inf")

            if top_k is not None and top_k > 0:
                values, _ = torch.topk(next_logits, k=min(top_k, next_logits.size(-1)))
                cutoff = values[:, [-1]]
                next_logits = torch.where(
                    next_logits < cutoff,
                    torch.full_like(next_logits, float("-inf")),
                    next_logits,
                )

            if top_p is not None and 0.0 < top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(next_logits, descending=True, dim=-1)
                sorted_probs = torch.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                sorted_remove = cumulative_probs > top_p
                sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
                sorted_remove[..., 0] = False

                remove_mask = torch.zeros_like(next_logits, dtype=torch.bool)
                remove_mask.scatter_(1, sorted_indices, sorted_remove)
                next_logits = next_logits.masked_fill(remove_mask, float("-inf"))

            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids


model_cfg = MiniGPTConfig(
    vocab_size=hf_tokenizer.vocab_size,
    context_length=CONFIG["context_length"],
    embedding_dim=CONFIG["embedding_dim"],
    num_heads=CONFIG["num_heads"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
)

model = MiniGPT(model_cfg).to(DEVICE)
param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Total parameters: {param_count:,}")
print(f"Trainable parameters: {trainable_count:,}")

x_test = torch.randint(0, model_cfg.vocab_size, (2, model_cfg.context_length), device=DEVICE)
logits_test, loss_test = model(x_test, x_test)
print("Forward shape test:", logits_test.shape, "| loss:", float(loss_test.detach().cpu()))

## 11. Training Utilities (Loss, Optimizer, Scheduler, AMP)

We set up AdamW, LR scheduling, gradient clipping, and automatic mixed precision for Colab GPU speedup.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

steps_per_epoch = len(train_loader)
updates_per_epoch = math.ceil(steps_per_epoch / CONFIG["grad_accum_steps"])
planned_total_updates = CONFIG["epochs"] * updates_per_epoch
if CONFIG.get("max_steps") is not None:
    planned_total_updates = min(planned_total_updates, int(CONFIG["max_steps"]))

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Updates per epoch (after grad accumulation): {updates_per_epoch}")
print(f"Planned total optimizer updates: {planned_total_updates}")
if planned_total_updates > 15000:
    print("Warning: this is a large number of updates for Kaggle. Reduce tokens, epochs, or increase stride.")


def lr_lambda(step: int) -> float:
    if step < CONFIG["warmup_steps"]:
        return float(step + 1) / float(max(1, CONFIG["warmup_steps"]))
    progress = (step - CONFIG["warmup_steps"]) / float(max(1, planned_total_updates - CONFIG["warmup_steps"]))
    return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def amp_autocast():
    if USE_AMP:
        return torch.amp.autocast(device_type="cuda")
    return contextlib.nullcontext()


@torch.no_grad()
def evaluate(model: MiniGPT, loader: DataLoader, max_batches: int) -> tuple[float, float]:
    model.eval()
    losses = []
    for idx, (x, y) in enumerate(loader):
        if idx >= max_batches:
            break
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with amp_autocast():
            _, loss = model(x, y)
        losses.append(float(loss.detach().cpu()))

    mean_loss = float(sum(losses) / max(1, len(losses)))
    perplexity = float(math.exp(min(20.0, mean_loss)))
    model.train()
    return mean_loss, perplexity

## 12. Colab GPU Training Loop with Checkpointing

This loop trains Mini-GPT, evaluates periodically, and saves `best.pt` + `last.pt` checkpoints.

In [ ]:
history = {
    "epoch": [],
    "step": [],
    "train_loss": [],
    "val_loss": [],
    "val_ppl": [],
    "lr": [],
    "time_sec": [],
    "gpu_mem_mb": [],
}

best_val_loss = float("inf")
start_time = time.time()
global_update_step = 0

model.train()
optimizer.zero_grad(set_to_none=True)

pbar = tqdm(total=planned_total_updates, desc="Training updates")

for epoch in range(1, CONFIG["epochs"] + 1):
    if global_update_step >= planned_total_updates:
        break

    epoch_progress = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)

    for batch_idx, (x, y) in enumerate(epoch_progress, start=1):
        if global_update_step >= planned_total_updates:
            break

        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with amp_autocast():
            _, loss = model(x, y)
            loss_for_backprop = loss / CONFIG["grad_accum_steps"]

        scaler.scale(loss_for_backprop).backward()

        do_update = (batch_idx % CONFIG["grad_accum_steps"] == 0) or (batch_idx == len(train_loader))
        if do_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_update_step += 1
            pbar.update(1)

            current_train_loss = float(loss.detach().cpu())

            if (
                global_update_step % CONFIG["eval_interval"] == 0
                or global_update_step == 1
                or global_update_step == planned_total_updates
            ):
                val_loss, val_ppl = evaluate(model, val_loader, CONFIG["eval_batches"])
                elapsed = time.time() - start_time
                current_lr = optimizer.param_groups[0]["lr"]
                gpu_mem_mb = 0.0
                if torch.cuda.is_available():
                    gpu_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

                history["epoch"].append(epoch)
                history["step"].append(global_update_step)
                history["train_loss"].append(current_train_loss)
                history["val_loss"].append(val_loss)
                history["val_ppl"].append(val_ppl)
                history["lr"].append(current_lr)
                history["time_sec"].append(elapsed)
                history["gpu_mem_mb"].append(gpu_mem_mb)

                pbar.set_postfix(
                    {
                        "epoch": epoch,
                        "train_loss": f"{current_train_loss:.4f}",
                        "val_loss": f"{val_loss:.4f}",
                        "val_ppl": f"{val_ppl:.2f}",
                        "lr": f"{current_lr:.2e}",
                    }
                )

                ckpt_payload = {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "config": CONFIG,
                    "model_config": asdict(model_cfg),
                    "step": global_update_step,
                    "epoch": epoch,
                    "history": history,
                    "tokenizer_dir": TOKENIZER_DIR.as_posix(),
                }

                torch.save(ckpt_payload, CHECKPOINT_DIR / "last.pt")
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(ckpt_payload, CHECKPOINT_DIR / "best.pt")

pbar.close()
print("Training finished.")
print("Best val loss:", round(best_val_loss, 4))
print("Checkpoints:", list(CHECKPOINT_DIR.glob("*.pt")))

## 13. Validation Metrics and Perplexity Tracking

We compute final validation/test metrics and visualize training behavior.

In [ ]:
best_ckpt = torch.load(CHECKPOINT_DIR / "best.pt", map_location=DEVICE)
model.load_state_dict(best_ckpt["model_state_dict"])

val_loss, val_ppl = evaluate(model, val_loader, max_batches=CONFIG["eval_batches"])
test_loss, test_ppl = evaluate(model, test_loader, max_batches=CONFIG["eval_batches"])

print(f"Validation loss: {val_loss:.4f} | Validation perplexity: {val_ppl:.2f}")
print(f"Test loss:       {test_loss:.4f} | Test perplexity:       {test_ppl:.2f}")

hist_df = pd.DataFrame(history)
display(hist_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df["step"], hist_df["train_loss"], label="train")
axes[0].plot(hist_df["step"], hist_df["val_loss"], label="val")
axes[0].set_title("Loss vs Step")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(hist_df["step"], hist_df["val_ppl"], label="val perplexity", color="tab:orange")
axes[1].set_title("Validation Perplexity vs Step")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Perplexity")
axes[1].legend()

plt.tight_layout()
plt.show()

## 14. Text Generation Testing (Greedy, Top-k, Top-p)

We compare decoding strategies and discuss quality differences during presentation.

In [ ]:
def generate_text(
    model: MiniGPT,
    tokenizer: PreTrainedTokenizerFast,
    prompt: str,
    max_new_tokens: int = 120,
    temperature: float = 0.9,
    top_k: int | None = 40,
    top_p: float | None = 0.92,
    repetition_penalty: float = 1.18,
    no_repeat_ngram_size: int = 3,
) -> str:
    model.eval()
    input_ids = tokenizer.encode(prompt, add_special_tokens=True)
    x = torch.tensor([input_ids], dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        out = model.generate(
            x,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )

    return tokenizer.decode(out[0].tolist(), skip_special_tokens=True)

prompts = [
    "A curious boy built his first AI model and",
    "The teacher asked the class to imagine a smart city where",
    "When the astronaut landed on Mars, she discovered",
]

rows = []
for p in prompts:
    greedy = generate_text(model, hf_tokenizer, p, max_new_tokens=100, temperature=0.75, top_k=1, top_p=None)
    topk = generate_text(model, hf_tokenizer, p, max_new_tokens=100, temperature=0.9, top_k=50, top_p=None)
    topp = generate_text(model, hf_tokenizer, p, max_new_tokens=100, temperature=0.9, top_k=None, top_p=0.92)
    rows.append({"prompt": p, "greedy_or_top1": greedy, "top_k": topk, "top_p": topp})

gen_df = pd.DataFrame(rows)
display(gen_df)

## 15. Model and Tokenizer Save/Load for Demo

We save model + tokenizer + configs, reload in a fresh cell, and verify inference so your live demo is robust.

In [ ]:
export_dir = ARTIFACT_DIR / "mini_gpt_demo_export"
export_dir.mkdir(parents=True, exist_ok=True)

model_export_path = export_dir / "mini_gpt_state.pt"
config_export_path = export_dir / "mini_gpt_config.json"
train_config_export_path = export_dir / "training_config.json"
tokenizer_export_dir = export_dir / "tokenizer"

# Save final artifacts
torch.save(model.state_dict(), model_export_path)
with open(config_export_path, "w", encoding="utf-8") as f:
    json.dump(asdict(model_cfg), f, indent=2)
with open(train_config_export_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)
hf_tokenizer.save_pretrained(tokenizer_export_dir.as_posix())

print("Saved model and tokenizer artifacts to:", export_dir)
for p in export_dir.rglob("*"):
    if p.is_file():
        print(" -", p.relative_to(export_dir))

# Reload for demo validation
with open(config_export_path, "r", encoding="utf-8") as f:
    reloaded_model_cfg = MiniGPTConfig(**json.load(f))

reloaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_export_dir.as_posix())
reloaded_model = MiniGPT(reloaded_model_cfg).to(DEVICE)
reloaded_model.load_state_dict(torch.load(model_export_path, map_location=DEVICE))
reloaded_model.eval()

demo_prompt = "The little girl found a hidden map and"
demo_out = generate_text(
    reloaded_model,
    reloaded_tokenizer,
    demo_prompt,
    max_new_tokens=90,
    temperature=0.9,
    top_k=30,
)
print("Prompt:", demo_prompt)
print("\nGenerated:\n", demo_out)

## 16. Demo Cells for Presentation Prompts and Outputs

These final cells are compact and ready for live presentation usage.

In [ ]:
presentation_prompts = [
    "A curious boy built his first AI model and",
    "The teacher asked the class to imagine a smart city where",
    "When the astronaut landed on Mars, she discovered",
]

for i, prompt in enumerate(presentation_prompts, 1):
    print("=" * 90)
    print(f"Prompt {i}: {prompt}")
    output = generate_text(
        reloaded_model,
        reloaded_tokenizer,
        prompt,
        max_new_tokens=80,
        temperature=0.9,
        top_k=30,
        top_p=0.9,
    )
    print("Output:")
    print(output)
    print()

In [ ]:
import shutil

# ── Save artifacts to Google Drive ────────────────────────────────────────────
# Run this cell after training finishes to keep your model safe.
# Colab sessions reset on disconnect — Google Drive persists across sessions.

from google.colab import drive
drive.mount("/content/drive")

drive_export_dir = Path("/content/drive/MyDrive/mini_gpt_demo_export")
drive_export_dir.mkdir(parents=True, exist_ok=True)

dest = drive_export_dir / "run_01"
shutil.copytree(export_dir, dest, dirs_exist_ok=True)

print("Artifacts saved to Google Drive:", dest)
print("Files saved:")
for p in dest.rglob("*"):
    if p.is_file():
        print(" -", p.relative_to(dest))


## Kaggle and Colab Run Profiles

### Why we switched from TinyStories to OpenWebText

TinyStories trains fast and produces grammatical text, but every prompt collapses into "Once upon a time" story mode regardless of topic. That happens because the domain is too narrow — the model only ever saw children's stories.

**OpenWebText** (`Skylion007/openwebtext`) is the web-scraped corpus used to train GPT-2. It contains news, technology, science, health, and general web content. Prompts about AI, smart cities, and astronauts will get contextually grounded continuations instead of fairy-tale responses.

---

### Recommended profile — 1 to 1.5 hours on Colab T4 / Kaggle P100

```python
"selected_dataset_id": "openwebtext"
"max_train_tokens": 8_000_000
"max_val_tokens":   400_000
"max_test_tokens":  400_000
"vocab_size":       16000
"context_length":   256
"sequence_stride":  128
"embedding_dim":    384
"num_layers":       6
"num_heads":        6
"batch_size":       16
"grad_accum_steps": 2
"epochs":           3
```

This keeps the model at ~20M parameters and trains on 8M tokens with stride-128 packing.

---

### If training runs over time — reduce one at a time

1. `"max_train_tokens": 6_000_000` (biggest impact)
2. `"epochs": 2`
3. `"sequence_stride": 192` (fewer samples, faster)

### If you want even better quality (2+ hours available)

- `"max_train_tokens": 12_000_000`
- `"epochs": 4`
- `"embedding_dim": 512` and `"num_heads": 8` (model grows to ~30M params)

---

### Dataset comparison for your prompts

| Prompt style | Best dataset |
|---|---|
| AI, tech, cities, science | **OpenWebText** |
| Creative children's stories | TinyStories |
| Wikipedia/factual text | wikitext-103 |

---

### Decode settings for quality outputs

```python
temperature = 0.85        # lower = more coherent, less random
top_k = 40                # restricts to likely tokens
top_p = 0.92              # nucleus sampling
repetition_penalty = 1.15 # penalises seen tokens
no_repeat_ngram_size = 3  # blocks 3-gram repetition loops
```

---

### Talking points for presentation

- OpenWebText is class-equivalent to GPT-2 pretraining data — good talking point.
- Sequence stride controls how many training samples are built from the token stream.
- With 20M parameters and 8M tokens, this is a small but real language model.
- Val perplexity below ~50 on web text indicates the model is learning meaningful patterns.
